In [3]:
from dotenv import load_dotenv
import os
from openai import OpenAI
from pypdf import PdfReader
from IPython.display import Markdown, display
import gradio as gr
import json
load_dotenv(override=True)
openai = OpenAI(base_url=os.getenv("OLLAMA_HOST"),api_key="ollama")


c:\Users\SHASHWAT\Shashwat_pal\GitHub\TanStack_project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:

summary = "shashwat is a software engineer working at google and pay role based on concentrix."
linkedin = "my linked id is shashwatpal1021 have a 5 years of the experience"

system_prompt = f"""

# Your role

You are a digital twin running on a website, chatting with visitors of the website.
You represent the person who's website you are on.
You answer questions related to their career, background, skills and experience.

Here are the details of the person you are representing:

{summary}

If asked, you explain clearly that you are an AI that is the digital twin of this person.

# Context

Here is a summary of the person's LinkedIn profile so that you can answer questions:

{linkedin}

# Rules

Engage with the user. Be professional and engaging, as if talking to a potential client or future employer who came across the website.
Avoid answering questions that are not related to the user's career, background, skills and experience;
steer the conversation back to professional topics.

Always stay in character as the digital twin of the person you are representing. Represent the person.

IMPORTANT: If you don't know the answer, say so. Never make up an answer.
If the user asks about something not in the context, say that you don't know.
"""

## Tools now

In [5]:
def record_email_tool(email):
    print(f"Tool called to record an email: {email}")
    with open("email.txt","a",encoding="utf-8") as f:
        f.write(email+"\n")
    return "Email received"

record_email_tool("shashwat@testy.com")

Tool called to record an email: shashwat@testy.com


'Email received'

In [6]:
# Steps 1- writw some json to describe the tool

record_email_tool_json = {
    "name":"record_email_tool",
    "description":"Use this tool to record that user provided their email address",
    "parameters":{
        "type":"object",
        "properties":{
            "email":{"type":"string","description":"The email address of this user "}
        },
        "required":["email"],
        "additionalProperties":False
    }
}

tools = [{"type":"function","function":record_email_tool_json}]
tools

[{'type': 'function',
  'function': {'name': 'record_email_tool',
   'description': 'Use this tool to record that user provided their email address',
   'parameters': {'type': 'object',
    'properties': {'email': {'type': 'string',
      'description': 'The email address of this user '}},
    'required': ['email'],
    'additionalProperties': False}}}]

In [7]:
# step 2 -  a new chat() function

"""
This is where we implement the tool call.
The reality is, it's a bit clunky. this is like seeing the ingredients of a fine recipem and finding that the ingredient turn out to be quite ordinary.

tool calling is an if statement. in this case, we are hardcoding everything to assume that the only tool is an email tool.


"""

"\nThis is where we implement the tool call.\nThe reality is, it's a bit clunky. this is like seeing the ingredients of a fine recipem and finding that the ingredient turn out to be quite ordinary.\n\ntool calling is an if statement. in this case, we are hardcoding everything to assume that the only tool is an email tool.\n\n\n"

In [13]:
import os, json

def responses(messages, tools=None):
    response = openai.chat.completions.create(
        model=os.getenv("MODEL_NAME"),
        messages=messages,
        tools=tools
    )
    return response


def chat(message, history):
    # Build message list
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    
    # First response
    response = responses(messages, tools)

    # Handle tool calls
    if response.choices[0].finish_reason == "tool_calls":
        message_obj = response.choices[0].message
        tool_call = message_obj.tool_calls[0]
        email = json.loads(tool_call.function.arguments).get("email")
        
        record_email_tool(email)
        
        messages.append(message_obj)
        messages.append({
            "role": "tool",
            "content": "Email recorded",
            "tool_call_id": tool_call.id
        })
        
        # Second response after tool execution
        response = responses(messages, tools)
    
    return response.choices[0].message.content


In [ ]:
gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


Tool called to record an email: shashwatpal1021
Tool called to record an email: shashwat_pal1021


In [15]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = responses(messages,tools)
         
    while response.choices[0].finish_reason=="tool_calls":
            message = response.choices[0].message
            messages.append(message)
            for tool_call in message.tool_calls:
                email = json.loads(tool_call.function.arguments).get("email")
                record_email_tool(email)
                messages.append({"role": "tool", "content": "Email recorded", "tool_call_id": tool_call.id})
            response = responses(messages,tools)
            
    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


Tool called to record an email: shashwatpal1021
Tool called to record an email: shashwat.cse.2022@gmail.com
